# inspect_quantization_surface
Establishes, from source, what OmniGen2 ships for quantized training (nothing), whether 4-bit loading is reachable through the model class it does ship, and which modules NF4 would touch against the four LoRA adapts. Companion to `inspect_official_training` and `inspect_soft_prompt_surface`.

QLoRA here means the diffusion transformer loaded in NF4 with the LoRA adapter unchanged. The MLLM, VAE and text encoder stay bf16: the MLLM is frozen and shared with the prompting arm, so quantizing it would move the conditioning hidden states and confound every comparison upstream.

Sections 4-6 print the exact lines a QLoRA patch would modify. Section 7 records the FSDP interaction, which is the one issue that can corrupt a run without raising. Read-only: clones the repo, instantiates the transformer on the meta device from its config, writes nothing.

## 1. Clone (skip if present)

In [ ]:
import os, subprocess, sys, re, json, textwrap
REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","--depth","1","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
print(subprocess.run(["git","-C",REPO,"rev-parse","--short","HEAD"], capture_output=True, text=True).stdout.strip(), "= repo commit")

18e6f9d = repo commit


## 2. Print helpers

In [ ]:
def head(t): print("\n" + "=" * 100 + f"\n{t}\n" + "=" * 100)

def show(relpath, first=None, last=None, grep=None, ctx=2):
    p = os.path.join(REPO, relpath)
    if not os.path.exists(p):
        print(f"  [absent] {relpath}"); return []
    lines = open(p, encoding="utf-8").read().split("\n")
    if grep:
        hits = [i for i, l in enumerate(lines) if re.search(grep, l)]
        for i in hits:
            for j in range(max(0, i - ctx), min(len(lines), i + ctx + 1)):
                print(f"  {relpath}:{j+1:>4}  {lines[j]}")
            print("  " + "-" * 60)
        return hits
    sel = lines[(first or 1) - 1:(last or len(lines))]
    for k, l in enumerate(sel, start=(first or 1)):
        print(f"  {relpath}:{k:>4}  {l}")
    return list(range(len(sel)))

## 3. Installed versions
`requirements.txt` pins torch and transformers but leaves diffusers unpinned, so the quantization surface is whatever the session resolves. Recorded here because the patch in section 5 depends on it.

In [ ]:
import importlib
head("versions")
for m in ["torch","diffusers","transformers","peft","accelerate","bitsandbytes"]:
    try:
        mod = importlib.import_module(m)
        print(f"  {m:14s} {getattr(mod, '__version__', '?')}")
    except ImportError:
        print(f"  {m:14s} NOT INSTALLED")
head("requirements.txt")
show("requirements.txt")


versions
  torch          2.11.0+cpu
  diffusers      0.39.0
  transformers   5.13.1
  peft           0.19.1
  accelerate     1.14.0
  bitsandbytes   NOT INSTALLED

requirements.txt
  requirements.txt:   1  torch==2.6.0
  requirements.txt:   2  torchvision==0.21.0
  requirements.txt:   3  timm
  requirements.txt:   4  einops
  requirements.txt:   5  accelerate
  requirements.txt:   6  transformers==4.51.3
  requirements.txt:   7  diffusers
  requirements.txt:   8  opencv-python-headless
  requirements.txt:   9  scipy
  requirements.txt:  10  wandb
  requirements.txt:  11  matplotlib
  requirements.txt:  12  Pillow
  requirements.txt:  13  tqdm
  requirements.txt:  14  omegaconf
  requirements.txt:  15  python-dotenv
  requirements.txt:  16  ninja
  requirements.txt:  17  ipykernel
  requirements.txt:  18  wheel
  requirements.txt:  19  triton-windows; sys_platform == "win32"
  requirements.txt:  20  


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

## 4. What ships for quantization
Every reference to quantization in the training path. `use_8bit_adam` quantizes optimizer state, not weights, and is not QLoRA.

In [ ]:
head("train.py — quantization references")
q = show("train.py", grep=r"quant|bnb|bitsandbytes|4bit|4-bit|nf4|int8|8bit|load_in_", ctx=1)
head("options/ — every training config")
for f in sorted(os.listdir(os.path.join(REPO, "options"))):
    print(f"  options/{f}")
head("docs/FINETUNE.md — supported modes")
show("docs/FINETUNE.md", grep=r"LoRA|full-parameter|Full-Parameter|quant", ctx=0)

src = open(os.path.join(REPO, "train.py"), encoding="utf-8").read()
weight_quant = re.findall(r"^.*(?:load_in_4bit|load_in_8bit|BitsAndBytesConfig|quantization_config).*$", src, re.M)
print(f"\n  weight-quantization references in train.py: {len(weight_quant)}")
assert not weight_quant, "train.py already references weight quantization; the patch below is redundant"


train.py — quantization references
  train.py: 282      # Use 8-bit Adam for lower memory usage or to fine-tune the model in 16GB GPUs
  train.py: 283      if args.train.use_8bit_adam:
  train.py: 284          try:
  ------------------------------------------------------------
  train.py: 284          try:
  train.py: 285              import bitsandbytes as bnb
  train.py: 286          except ImportError:
  ------------------------------------------------------------
  train.py: 287              raise ImportError(
  train.py: 288                  "To use 8-bit Adam, please install the bitsandbytes library: `pip install bitsandbytes`."
  train.py: 289              )
  ------------------------------------------------------------
  train.py: 290  
  train.py: 291          optimizer_class = bnb.optim.AdamW8bit
  train.py: 292      else:
  ------------------------------------------------------------

options/ — every training config
  options/ft.yml
  options/ft_lora.yml

docs/FINETUNE.md 

## 5. The construction site
The single anchor a QLoRA patch replaces. Printed verbatim so the `src.replace` target can be copied exactly, in the pattern the LoRA and soft-prompt notebooks use.

In [ ]:
head("train.py — transformer construction")
show("train.py", grep=r"OmniGen2Transformer2DModel\.from_pretrained", ctx=4)
head("train.py — text encoder and VAE construction (stay bf16)")
show("train.py", grep=r"TextEncoder\.from_pretrained|AutoencoderKL|vae\s*=", ctx=2)


train.py — transformer construction
  train.py: 197          return model
  train.py: 198      
  train.py: 199      ema_decay = args.train.get('ema_decay', 0)
  train.py: 200  
  train.py: 201      model = OmniGen2Transformer2DModel.from_pretrained(
  train.py: 202          args.model.pretrained_model_path, subfolder="transformer"
  train.py: 203      )
  train.py: 204      model.train()
  train.py: 205  
  ------------------------------------------------------------

train.py — text encoder and VAE construction (stay bf16)
  train.py:  41  from diffusers.optimization import get_scheduler
  train.py:  42  from diffusers.utils.torch_utils import is_compiled_module
  train.py:  43  from diffusers.models.autoencoders.autoencoder_kl import AutoencoderKL
  train.py:  44  
  train.py:  45  from peft import LoraConfig
  ------------------------------------------------------------
  train.py: 232          text_tokenizer.save_pretrained(os.path.join(args.output_dir, 'tokenizer'))
  train.py: 

[42, 233, 245, 254]

## 6. Is 4-bit reachable
`from_pretrained` accepts `quantization_config` only if the class inherits diffusers' `ModelMixin`. Checked against the class declaration and the resolved signature rather than assumed.

In [ ]:
head("model class declaration")
show("omnigen2/models/transformers/transformer_omnigen2.py",
     grep=r"^class OmniGen2Transformer2DModel", ctx=0)

sys.path.insert(0, REPO)
from omnigen2.models.transformers.transformer_omnigen2 import OmniGen2Transformer2DModel
import inspect as _inspect
from diffusers.models.modeling_utils import ModelMixin

mro = [c.__name__ for c in OmniGen2Transformer2DModel.__mro__]
print("\n  MRO:", " -> ".join(mro[:6]))
print(f"  inherits ModelMixin: {issubclass(OmniGen2Transformer2DModel, ModelMixin)}")
assert issubclass(OmniGen2Transformer2DModel, ModelMixin), "no ModelMixin; quantization_config is not reachable"

sig = _inspect.signature(ModelMixin.from_pretrained)
accepts_kwargs = any(p.kind is _inspect.Parameter.VAR_KEYWORD for p in sig.parameters.values())
print(f"  from_pretrained parameters: {list(sig.parameters)[:6]}{' ...' if len(sig.parameters) > 6 else ''}")
print(f"  accepts **kwargs (quantization_config routes through this): {accepts_kwargs}")

try:
    from diffusers.quantizers.quantization_config import BitsAndBytesConfig as DiffusersBnBConfig
    print("  diffusers BitsAndBytesConfig: available")
except ImportError as e:
    print(f"  diffusers BitsAndBytesConfig: NOT AVAILABLE ({e})")
    print("  -> the installed diffusers predates model-level quantization; pin a newer one")


model class declaration
  omnigen2/models/transformers/transformer_omnigen2.py: 222  class OmniGen2Transformer2DModel(ModelMixin, ConfigMixin, PeftAdapterMixin, FromOriginalModelMixin):
  ------------------------------------------------------------



  MRO: OmniGen2Transformer2DModel -> ModelMixin -> Module -> PushToHubMixin -> ConfigMixin -> PeftAdapterMixin
  inherits ModelMixin: True
  from_pretrained parameters: ['pretrained_model_name_or_path', 'kwargs']
  accepts **kwargs (quantization_config routes through this): True
  diffusers BitsAndBytesConfig: available


/content/OmniGen2/omnigen2/models/attention_processor.py:33: UserWarning: Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance")
/content/OmniGen2/omnigen2/models/transformers/block_lumina2.py:31: UserWarning: Cannot import triton, install triton to use fused RMSNorm for better performance
  warnings.warn("Cannot import triton, install triton to use fused RMSNorm for better performance")
/content/OmniGen2/omnigen2/models/transformers/block_lumina2.py:37: UserWarning: Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance")


## 7. Module inventory
Which `nn.Linear` modules NF4 would replace against the four the LoRA config adapts. Instantiated on the meta device from the released config, so no weights are downloaded.

In [ ]:
import torch
from huggingface_hub import hf_hub_download

CFG = hf_hub_download("OmniGen2/OmniGen2", "transformer/config.json")
cfg = json.load(open(CFG))
print(f"  config keys: {sorted(cfg)[:10]} ...")

with torch.device("meta"):
    m = OmniGen2Transformer2DModel.from_config(cfg)

lin = [(n, mod.in_features, mod.out_features)
       for n, mod in m.named_modules() if isinstance(mod, torch.nn.Linear)]
TARGETS = ["to_k", "to_q", "to_v", "to_out.0"]          # train.py LoraConfig.target_modules
is_target = lambda n: any(n.endswith(t) for t in TARGETS)

adapted = [x for x in lin if is_target(x[0])]
other   = [x for x in lin if not is_target(x[0])]
print(f"\n  nn.Linear total            : {len(lin)}")
print(f"  adapted by LoRA            : {len(adapted)}  ({len(adapted)//4} attention blocks x 4 projections)")
print(f"  quantized but not adapted  : {len(other)}")

head("modules LoRA adapts (first 8)")
for n, i, o in adapted[:8]: print(f"  {n:66s} {i:>5} -> {o:>5}")
head("modules NF4 would also quantize, grouped by leaf name")
import collections
leaf = collections.Counter(n.split(".")[-1] for n, _, _ in other)
for k, v in leaf.most_common(): print(f"  {k:28s} {v}")
head("non-adapted modules outside the repeated block stacks (skip-module candidates)")
stacks = ("layers.", "noise_refiner.", "ref_image_refiner.", "context_refiner.")
for n, i, o in other:
  if not any(s in n for s in stacks): print(f"  {n:66s} {i:>5} -> {o:>5}")

params_lin = sum(i * o for _, i, o in lin)
params_all = sum(p.numel() for p in m.parameters())
print(f"\n  transformer parameters     : {params_all:,}")
print(f"  in nn.Linear weights       : {params_lin:,}  ({100*params_lin/params_all:.1f}%)")
print(f"  bf16 footprint             : {params_all*2/2**30:.2f} GiB")
print(f"  NF4 footprint (linear only): {(params_lin*0.5 + (params_all-params_lin)*2)/2**30:.2f} GiB")

config.json:   0%|          | 0.00/494 [00:00<?, ?B/s]

  config keys: ['_class_name', '_diffusers_version', 'axes_dim_rope', 'axes_lens', 'ffn_dim_multiplier', 'hidden_size', 'in_channels', 'multiple_of', 'norm_eps', 'num_attention_heads'] ...

  nn.Linear total            : 309
  adapted by LoRA            : 152  (38 attention blocks x 4 projections)
  quantized but not adapted  : 157

modules LoRA adapts (first 8)
  noise_refiner.0.attn.to_q                                           2520 ->  2520
  noise_refiner.0.attn.to_k                                           2520 ->   840
  noise_refiner.0.attn.to_v                                           2520 ->   840
  noise_refiner.0.attn.to_out.0                                       2520 ->  2520
  noise_refiner.1.attn.to_q                                           2520 ->  2520
  noise_refiner.1.attn.to_k                                           2520 ->   840
  noise_refiner.1.attn.to_v                                           2520 ->   840
  noise_refiner.1.attn.to_out.0                

## 8. The LoRA config to hold fixed
Reproduced from `train.py` so the QLoRA run adapts exactly what the LoRA run adapted. `lora_alpha == lora_rank` makes the adapter scale 1 at every rank, which is what keeps the LoRA arm's rank-equivalence result from being an alpha artifact.

In [ ]:
head("train.py — LoraConfig")
show("train.py", grep=r"LoraConfig|target_modules|lora_alpha|init_lora_weights|lora_ft", ctx=1)


train.py — LoraConfig
  train.py:  44  
  train.py:  45  from peft import LoraConfig
  train.py:  46  
  ------------------------------------------------------------
  train.py: 257      
  train.py: 258      args.train.lora_ft = args.train.get('lora_ft', False)
  train.py: 259      if args.train.lora_ft:
  ------------------------------------------------------------
  train.py: 258      args.train.lora_ft = args.train.get('lora_ft', False)
  train.py: 259      if args.train.lora_ft:
  train.py: 260          model.requires_grad_(False)
  ------------------------------------------------------------
  train.py: 261  
  train.py: 262          target_modules = ["to_k", "to_q", "to_v", "to_out.0"]
  train.py: 263  
  ------------------------------------------------------------
  train.py: 264          # now we will add new LoRA weights the transformer layers
  train.py: 265          lora_config = LoraConfig(
  train.py: 266              r=args.train.lora_rank,
  ---------------------------

[44, 257, 258, 261, 264, 266, 268, 269]

## 9. FSDP interaction
The LoRA runs launch with `--use_fsdp`. FSDP flattens parameters into uniform-dtype FlatParams; bitsandbytes stores 4-bit weights as a uint8 tensor with a separate `quant_state`. With `--fsdp_sync_module_states True` the parameters are broadcast, and `quant_state` can be dropped without raising, leaving a run that trains against corrupted weights and still reports finite losses.

On `--num_processes=1` with `HYBRID_SHARD_ZERO2` there are no peer ranks, so FSDP shards nothing and is inherited from the official launch script rather than doing work. The QLoRA smoke test establishes the non-FSDP path first and adds FSDP back only against a `quant_state` round-trip check.

`accelerator.clip_grad_norm_` dispatches to FSDP's implementation under FSDP and to the plain one without it. Same value on one process, different code path, and the gradient monitor reads that return value.

In [ ]:
head("clip_grad_norm_ call site")
show("train.py", grep=r"clip_grad_norm_", ctx=3)
head("loss normalisation anchor (A4) — patched identically in every arm")
show("train.py", grep=r"num_tokens_in_batch", ctx=1)


clip_grad_norm_ call site
  train.py: 589                  bin_sum_loss = accelerator.gather(rearrange(bin_sum_loss, "b -> 1 b")).sum(dim=0)
  train.py: 590  
  train.py: 591                  if accelerator.sync_gradients:
  train.py: 592                      accelerator.clip_grad_norm_(trainable_params, args.train.max_grad_norm)
  train.py: 593  
  train.py: 594                  optimizer.step()
  train.py: 595                  if 'timm' in args.train.lr_scheduler:
  ------------------------------------------------------------

loss normalisation anchor (A4) — patched identically in every arm
  train.py: 550  
  train.py: 551                  local_num_tokens_in_batch = 0
  train.py: 552  
  ------------------------------------------------------------
  train.py: 553                  for i, latent in enumerate(output_latents):
  train.py: 554                      local_num_tokens_in_batch += latent.numel()
  train.py: 555                  
  ------------------------------------------

[550, 553, 555, 565]

## 10. Summary

In [ ]:
head("summary")
print(f"""
  ships for quantized training : nothing; use_8bit_adam is optimizer state, not weights
  4-bit reachable              : via ModelMixin.from_pretrained(quantization_config=...)
  patch surface                : one anchor, train.py OmniGen2Transformer2DModel.from_pretrained
  quantize                     : diffusion transformer only
  leave bf16                   : MLLM, text encoder, VAE
  adapter                      : unchanged from the LoRA arm, alpha == rank
  loss normalisation           : FIX_ACCUM_NORM applied, same anchor as every other arm
  optimizer                    : AdamW, matching the LoRA arm, not paged 8-bit
  open question for the smoke test:
    FSDP + Params4bit          : quant_state survival under sync_module_states
    checkpoint format          : adapter-only save must load via
                                 pipe.transformer.load_lora_adapter(...)
""")


summary

  ships for quantized training : nothing; use_8bit_adam is optimizer state, not weights
  4-bit reachable              : via ModelMixin.from_pretrained(quantization_config=...)
  patch surface                : one anchor, train.py OmniGen2Transformer2DModel.from_pretrained
  quantize                     : diffusion transformer only
  leave bf16                   : MLLM, text encoder, VAE
  adapter                      : unchanged from the LoRA arm, alpha == rank
  loss normalisation           : FIX_ACCUM_NORM applied, same anchor as every other arm
  optimizer                    : AdamW, matching the LoRA arm, not paged 8-bit
  open question for the smoke test:
    FSDP + Params4bit          : quant_state survival under sync_module_states
    checkpoint format          : adapter-only save must load via
                                 pipe.transformer.load_lora_adapter(...)

